<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone --recursive --branch zeynep-september https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git \
/content/code

Cloning into '/content/code'...
remote: Enumerating objects: 903, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 903 (delta 98), reused 86 (delta 86), pack-reused 786 (from 2)
Receiving objects: 100% (903/903), 96.52 MiB | 34.22 MiB/s, done.
Resolving deltas: 100% (488/488), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 12.59 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'


In [3]:
%cd /content/code

/content/code


In [4]:
import sys
import torch
import torchvision

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
CUDA version: 12.8


In [5]:
!pip install -q torcheval pyrebase4 yacs loguru wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.


In [6]:
import os
import subprocess

DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"
FEATURES_ZIP = f"{DRIVE_BASE_PATH}/1s.zip"

CODE_DIR = "/content/code"
TEMP_DIR = "/content/temp_features"
VIDEO_DIR = f"{CODE_DIR}/data/video"

# clean temp area
!rm -rf "{TEMP_DIR}"
!mkdir -p "{TEMP_DIR}"
!mkdir -p "{VIDEO_DIR}"

# copy the outer zip locally
!cp "{FEATURES_ZIP}" /content/1s.zip

# unzip outer archive
!unzip -q -o /content/1s.zip -d "{TEMP_DIR}"

# omnivore.zip already contains an omnivore/ folder, so extract straight
# into data/video -> data/video/omnivore, which is where the dataloader
# expects features (segment_features_directory="data/" + "video" + backbone)
!unzip -q -o "{TEMP_DIR}/1s/video/omnivore.zip" -d "{VIDEO_DIR}"

# clean temp files
!rm /content/1s.zip
!rm -rf "{TEMP_DIR}"

print("Omnivore features ready.")

Omnivore features ready.


In [7]:
files = os.listdir("/content/code/data/video/omnivore")
print("Number of Omnivore feature files:", len(files))
print(files[:5])


Number of Omnivore feature files: 384
['29_5_360p.mp4_1s_1s.npz', '28_26_360p.mp4_1s_1s.npz', '29_29_360p.mp4_1s_1s.npz', '13_24_360p.mp4_1s_1s.npz', '25_3_360p.mp4_1s_1s.npz']


In [8]:
DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"

CHECKPOINT_ZIP = f"{DRIVE_BASE_PATH}/error_recognition_best.zip"

In [9]:
!unzip -l "$CHECKPOINT_ZIP" | head -30

Archive:  /content/drive/MyDrive/AML_Project/error_recognition_best.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   825896  2024-05-20 17:51   error_recognition_best/MLP/3dresnet/error_recognition_MLP_3dresnet_recordings_epoch_45.pt
   825904  2024-05-20 19:41   error_recognition_best/MLP/3dresnet/error_recognition_MLP_3dresnet_environment_epoch_11.pt
   825784  2024-05-20 20:51   error_recognition_best/MLP/3dresnet/error_recognition_MLP_3dresnet_step_epoch_41.pt
   825800  2024-05-20 18:51   error_recognition_best/MLP/3dresnet/error_recognition_MLP_3dresnet_person_epoch_39.pt
  2103976  2024-05-21 19:16   error_recognition_best/MLP/imagebind/error_recognition_MLP_imagebind_audio_environment_epoch_50.pt
  2103864  2024-05-21 17:51   error_recognition_best/MLP/imagebind/error_recognition_MLP_imagebind_audio_person_epoch_8.pt
  2103856  2024-05-21 04:28   error_recognition_best/MLP/imagebind/error_recognition_MLP_imagebind_audio_step_epoch_28.pt
  2103960  20

In [10]:
import os

CHECKPOINT_DIR = "/content/code/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!cp "$CHECKPOINT_ZIP" /content/checkpoints.zip
!unzip -q -o /content/checkpoints.zip -d "$CHECKPOINT_DIR"
!rm /content/checkpoints.zip

print("Checkpoints extracted.")

Checkpoints extracted.


In [11]:
ckpt = "/content/code/checkpoints/error_recognition_best/MLP/omnivore/error_recognition_MLP_omnivore_step_epoch_43.pt"

print(os.path.exists(ckpt))

True


In [12]:
import os

print("annotations:", os.path.exists("/content/code/annotations/annotation_json/step_annotations.json"))
print("features:", os.path.exists("/content/code/data/video/omnivore"))
print("checkpoint:", os.path.exists("/content/code/checkpoints/error_recognition_best/MLP/omnivore/error_recognition_MLP_omnivore_step_epoch_43.pt"))

annotations: True
features: True
checkpoint: True


In [13]:
%cd /content/code
%pdb on

import sys
sys.path.insert(0, ".")

from core.evaluate import eval_er, Config
from constants import Constants as const

conf = Config()
conf.split = const.STEP_SPLIT
conf.backbone = const.OMNIVORE
conf.variant = const.MLP_VARIANT
conf.phase = const.TEST
conf.modality = const.VIDEO

conf.ckpt_directory = (
    "checkpoints/error_recognition_best/MLP/omnivore/"
    "error_recognition_MLP_omnivore_step_epoch_43.pt"
)

eval_er(conf, threshold=0.6)

/content/code
Automatic pdb calling has been turned ON
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


test Progress: 42347/798: 100%|██████████| 798/798 [00:06<00:00, 115.21it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.4096162736939436, 'recall': 0.2989708115404083, 'f1': 0.3456549302643129, 'accuracy': 0.6831416629277163, 'auc': np.float64(0.6541560352028618), 'pr_auc': tensor(0.3187)}
test Step Level Metrics: {'precision': 0.6607142857142857, 'recall': 0.14859437751004015, 'f1': 0.24262295081967214, 'accuracy': 0.7105263157894737, 'auc': np.float64(0.7573902166041213), 'pr_auc': tensor(0.3638)}
----------------------------------------------------------------


In [14]:
import subprocess
import re
import pandas as pd

experiments = [
    {
        "split": "step",
        "model": "MLP (Omnivore)",
        "variant": "MLP",
        "threshold": 0.6,
        "ckpt": "checkpoints/error_recognition_best/MLP/omnivore/"
                "error_recognition_MLP_omnivore_step_epoch_43.pt",
    },
    {
        "split": "recordings",
        "model": "MLP (Omnivore)",
        "variant": "MLP",
        "threshold": 0.4,
        "ckpt": "checkpoints/error_recognition_best/MLP/omnivore/"
                "error_recognition_MLP_omnivore_recordings_epoch_33.pt",
    },
    {
        "split": "step",
        "model": "Transformer (Omnivore)",
        "variant": "Transformer",
        "threshold": 0.6,
        "ckpt": "checkpoints/error_recognition_best/Transformer/omnivore/"
                "error_recognition_Transformer_omnivore_step_epoch_9.pt",
    },
    {
        "split": "recordings",
        "model": "Transformer (Omnivore)",
        "variant": "Transformer",
        "threshold": 0.4,
        "ckpt": "checkpoints/error_recognition_best/Transformer/omnivore/"
                "error_recognition_Transformer_omnivore_recordings_epoch_31.pt",
    },
]

results = []

for exp in experiments:
    print(f"\nRunning {exp['model']} — {exp['split']}")

    cmd = [
        "python", "-m", "core.evaluate",
        "--variant", exp["variant"],
        "--backbone", "omnivore",
        "--ckpt", exp["ckpt"],
        "--split", exp["split"],
        "--threshold", str(exp["threshold"]),
    ]

    process = subprocess.run(
        cmd,
        cwd="/content/code",
        capture_output=True,
        text=True
    )

    output = process.stdout + "\n" + process.stderr

    if process.returncode != 0:
        print("FAILED:")
        print(output)
        continue

    # Find all metrics blocks and use the final aggregate one
    metric_matches = re.findall(
        r"Metrics:\s*\{(.*?)\}",
        output
    )

    if not metric_matches:
        print("Could not find any metrics in output:")
        print(output)
        continue

    metrics_text = metric_matches[-1]

    precision_match = re.search(
        r"'precision':\s*([0-9.]+)",
        metrics_text
    )

    recall_match = re.search(
        r"'recall':\s*([0-9.]+)",
        metrics_text
    )

    f1_match = re.search(
        r"'f1':\s*([0-9.]+)",
        metrics_text
    )

    accuracy_match = re.search(
        r"'accuracy':\s*([0-9.]+)",
        metrics_text
    )

    auc_match = re.search(
        r"'auc':\s*(?:np\.float64\()?([0-9.]+)",
        metrics_text
    )

    if not all([
        precision_match,
        recall_match,
        f1_match,
        accuracy_match,
        auc_match
    ]):
        print("Could not parse all metrics from:")
        print(metrics_text)
        continue

    precision = float(precision_match.group(1)) * 100
    recall = float(recall_match.group(1)) * 100
    f1 = float(f1_match.group(1)) * 100
    accuracy = float(accuracy_match.group(1)) * 100
    auc = float(auc_match.group(1)) * 100

    results.append({
        "Split": exp["split"].capitalize(),
        "Model": exp["model"],
        "Accuracy": round(accuracy, 2),
        "Precision": round(precision, 2),
        "Recall": round(recall, 2),
        "F1": round(f1, 2),
        "AUC": round(auc, 2),
    })

df = pd.DataFrame(results)

df = df[
    ["Split", "Model", "Accuracy", "Precision", "Recall", "F1", "AUC"]
]

df


Running MLP (Omnivore) — step

Running MLP (Omnivore) — recordings

Running Transformer (Omnivore) — step

Running Transformer (Omnivore) — recordings


,Split,Model,Accuracy,Precision,Recall,F1,AUC
0,Step,MLP (Omnivore),71.05,66.07,14.86,24.26,75.74
1,Recordings,MLP (Omnivore),50.37,40.91,85.89,55.42,63.03
2,Step,Transformer (Omnivore),69.92,51.56,59.84,55.39,75.62
3,Recordings,Transformer (Omnivore),61.40,45.41,36.93,40.73,62.27
